## String Operations

### Why does this matter?

In real datasets, text columns are everywhere — customer names, product descriptions, city names, email addresses, status codes.

 Raw text is almost always dirty. Names have inconsistent casing, emails have trailing spaces, city names are misspelled, status codes 
 
 need to be extracted from longer strings. Cleaning and manipulating text is one of the most frequent tasks in real Data Analyst work.

### What is the str accessor?

### What is it?

In Pandas, every Series that contains strings has a special .str accessor. It gives you access to a whole collection of string methods 

that operate on every element of the column simultaneously — vectorized, just like arithmetic.

### Why does it exist?

Python strings have built-in methods like *.upper(), .replace(), .split().* 

But those work on a single string. 

You can't write *df['Name'].upper()* directly — that would fail because *upper()* doesn't know how to handle a Series.

 The .str accessor bridges that gap — it tells Pandas "apply this string method to every element in this column".

### How does it work internally?

*df['Name'].str.upper()* — Pandas iterates through each value in the column, calls *.upper()* on each string, and assembles the results back into a new Series.

 It handles NaN gracefully — NaN stays NaN instead of crashing.

In [2]:
import pandas as pd
import numpy as np

data = {
    'Name':    ['  sumed  ', 'PRIYA', 'Rahul K', 'neha',  'ARJUN'],
    'Email':   ['sumed@gmail.com', 'priya@yahoo.com',
                'rahul@gmail.com', 'neha@outlook.com', np.nan],
    'City':    ['Guwahati', 'Mumbai', 'Delhi', 'Pune', 'Bangalore'],
    'Code':    ['EMP-101-Data', 'EMP-102-HR', 'EMP-103-Data',
                'EMP-104-Finance', 'EMP-105-Data']
}
df = pd.DataFrame(data)

In [3]:
df

,Name,Email,City,Code
0,sumed,sumed@gmail.com,Guwahati,EMP-101-Data
1,PRIYA,priya@yahoo.com,Mumbai,EMP-102-HR
2,Rahul K,rahul@gmail.com,Delhi,EMP-103-Data
3,neha,neha@outlook.com,Pune,EMP-104-Finance
4,ARJUN,NaN,Bangalore,EMP-105-Data


*This is a realistic messy dataset — inconsistent casing, whitespace, NaN in strings, and encoded data packed into one column. All of this needs cleaning.*

## CASE conversion

In [4]:
# Case inconsistency is the most common string problem
# 'Data' != 'data' != 'DATA' — all treated differently in filters
# Always standardize case before filtering or groupby

df['Name'].str.lower()     # all lowercase




0      sumed  
1        priya
2      rahul k
3         neha
4        arjun
Name: Name, dtype: object

In [5]:
df['Name'].str.upper()     # all uppercase


0      SUMED  
1        PRIYA
2      RAHUL K
3         NEHA
4        ARJUN
Name: Name, dtype: object

In [6]:
df['Name'].str.title()     # First Letter Capitalized


0      Sumed  
1        Priya
2      Rahul K
3         Neha
4        Arjun
Name: Name, dtype: object

In [7]:
df['Name'].str.capitalize() # Only first letter of string

0      sumed  
1        Priya
2      Rahul k
3         Neha
4        Arjun
Name: Name, dtype: object

In [8]:
# Real world: standardize before groupby to avoid duplicate groups
df['City'] = df['City'].str.lower()
# Now 'Mumbai' and 'mumbai' won't create two separate groups

In [9]:
df

,Name,Email,City,Code
0,sumed,sumed@gmail.com,guwahati,EMP-101-Data
1,PRIYA,priya@yahoo.com,mumbai,EMP-102-HR
2,Rahul K,rahul@gmail.com,delhi,EMP-103-Data
3,neha,neha@outlook.com,pune,EMP-104-Finance
4,ARJUN,NaN,bangalore,EMP-105-Data


*Always call str.lower() or str.strip() before doing groupby() or filtering on text columns. Casing and whitespace are silent data quality killers.*

## STRIP : remove white space

In [10]:
# Whitespace (spaces, tabs, newlines) around values is invisible
# but breaks equality checks and groupby operations
# '  sumed  ' != 'sumed' — Python sees them as different strings

df['Name'].str.strip()    # removes spaces from BOTH sides


0      sumed
1      PRIYA
2    Rahul K
3       neha
4      ARJUN
Name: Name, dtype: object

In [11]:
df['Name'].str.lstrip()   # removes spaces from LEFT only


0    sumed  
1      PRIYA
2    Rahul K
3       neha
4      ARJUN
Name: Name, dtype: object

In [12]:
df['Name'].str.rstrip()   # removes spaces from RIGHT only



0      sumed
1      PRIYA
2    Rahul K
3       neha
4      ARJUN
Name: Name, dtype: object

In [13]:
# Standard cleaning pipeline — always do both
df['Name'] = df['Name'].str.strip().str.title()
# First strip whitespace, then fix casing — chained in one line

In [14]:
df

,Name,Email,City,Code
0,Sumed,sumed@gmail.com,guwahati,EMP-101-Data
1,Priya,priya@yahoo.com,mumbai,EMP-102-HR
2,Rahul K,rahul@gmail.com,delhi,EMP-103-Data
3,Neha,neha@outlook.com,pune,EMP-104-Finance
4,Arjun,NaN,bangalore,EMP-105-Data


## CONTAINS() / STARTSWITH() / ENDSWITH()

In [16]:
# str.contains() → returns True/False — use for filtering
# Like SQL's LIKE operator

# Find all gmail users
df[df['Email'].str.contains('gmail', na=False)]
# na=False → treat NaN as False instead of raising error
# Without na=False → NaN in Email column would cause TypeError


,Name,Email,City,Code
0,Sumed,sumed@gmail.com,guwahati,EMP-101-Data
2,Rahul K,rahul@gmail.com,delhi,EMP-103-Data


In [17]:

# Case-insensitive search
df[df['City'].str.contains('mumbai', case=False, na=False)]



,Name,Email,City,Code
1,Priya,priya@yahoo.com,mumbai,EMP-102-HR


In [18]:
# startswith / endswith
df[df['Code'].str.startswith('EMP')]   # all valid codes
df[df['Email'].str.endswith('.com', na=False)]

,Name,Email,City,Code
0,Sumed,sumed@gmail.com,guwahati,EMP-101-Data
1,Priya,priya@yahoo.com,mumbai,EMP-102-HR
2,Rahul K,rahul@gmail.com,delhi,EMP-103-Data
3,Neha,neha@outlook.com,pune,EMP-104-Finance


*⚠ Always pass na=False when using str.contains() or str.endswith() on columns that might have NaN. Without it, Pandas raises a ValueError when it hits the NaN.*

## REPLACE() : find and replace all strings

In [20]:
# str.replace(old, new) → replaces substring inside each string
# DIFFERENT from df.replace() which replaces whole values

# Remove 'EMP-' prefix from Code column
df['Code'].str.replace('EMP-', '', regex=False)
# regex=False → treat first argument as plain text, not regex pattern
# Always set regex=False for simple replacements — avoids surprises



0       101-Data
1         102-HR
2       103-Data
3    104-Finance
4       105-Data
Name: Code, dtype: object

In [21]:
# Replace multiple things by chaining
df['Code'].str.replace('EMP-', '', regex=False).str.replace('-', '_', regex=False)


0       101_Data
1         102_HR
2       103_Data
3    104_Finance
4       105_Data
Name: Code, dtype: object

In [22]:

# str.replace vs df['col'].replace
# str.replace → works INSIDE a string (substring replacement)
# df.replace  → replaces the WHOLE cell value
df['City'].replace('Mumbai', 'Bombay')   # whole value swap
df['City'].str.replace('mba', 'MBA')      # substring swap inside

0     guwahati
1       muMBAi
2        delhi
3         pune
4    bangalore
Name: City, dtype: object

## SPLIT() : break a strings into parts

In [24]:
# str.split(separator) → splits each string into a LIST
# Common use: extract parts from structured strings like 'EMP-101-Data'

# Split Code column on '-'
df['Code'].str.split('-')
# Returns: ['EMP', '101', 'Data'] for each row



0       [EMP, 101, Data]
1         [EMP, 102, HR]
2       [EMP, 103, Data]
3    [EMP, 104, Finance]
4       [EMP, 105, Data]
Name: Code, dtype: object

In [25]:
# expand=True → splits into separate columns automatically
df['Code'].str.split('-', expand=True)
# Returns a DataFrame with 3 columns: 0, 1, 2



,0,1,2
0,EMP,101,Data
1,EMP,102,HR
2,EMP,103,Data
3,EMP,104,Finance
4,EMP,105,Data


In [29]:
# Extract specific parts — assign to new columns
split_cols = df['Code'].str.split('-', expand=True)
df['EmpNum']   = split_cols[1]   # '101', '102', etc.
df['EmpDept']  = split_cols[2]   # 'Data', 'HR', etc.



In [30]:
# Extract email domain
df['Domain'] = df['Email'].str.split('@', expand=True)[1]
# [1] → takes second part after @  → 'gmail.com', 'yahoo.com'

In [31]:
df['Domain']

0      gmail.com
1      yahoo.com
2      gmail.com
3    outlook.com
4            NaN
Name: Domain, dtype: object

*expand=True is one of the most useful string tricks — turn one structured column into multiple clean columns. Interviewers love this pattern.*

## LEN()  COUNT()  FIND()

In [ ]:
# str.len() → length of each string
df['Name'].str.len()
# Use: filter out names that are too short (likely bad data)
df[df['Name'].str.len() > 3]

# str.count(sub) → count occurrences of substring
df['Email'].str.count('.', na=False)
# gmail.com → 1, yahoo.com → 1

# str.find(sub) → index position of first occurrence
# Returns -1 if not found
df['Email'].str.find('@')
# sumed@gmail.com → 5 (@ is at position 5)

# str.slice(start, end) → extract substring by position
df['Code'].str[4:7]   # extract characters 4,5,6 → '101', '102'...

## REAL WORLD CLEANING PIPELINE

In [33]:
# This is what a real data cleaning block looks like
# Every step has a reason

# 1. Strip whitespace from all string columns
df['Name']  = df['Name'].str.strip()

# 2. Standardize casing
df['Name']  = df['Name'].str.title()
df['City']  = df['City'].str.lower()
df['Email'] = df['Email'].str.lower()

# 3. Extract structured fields from Code column
split_df = df['Code'].str.split('-', expand=True)
df['EmpNum']  = split_df[1]
df['EmpDept'] = split_df[2]

# 4. Extract email provider
df['Provider'] = df['Email'].str.split('@', expand=True)[1]

# 5. Drop original Code column — no longer needed
df = df.drop(columns='Code')

In [34]:
df

,Name,Email,City,EmpNum,EmpDept,Domain,Provider
0,Sumed,sumed@gmail.com,guwahati,101,Data,gmail.com,gmail.com
1,Priya,priya@yahoo.com,mumbai,102,HR,yahoo.com,yahoo.com
2,Rahul K,rahul@gmail.com,delhi,103,Data,gmail.com,gmail.com
3,Neha,neha@outlook.com,pune,104,Finance,outlook.com,outlook.com
4,Arjun,NaN,bangalore,105,Data,NaN,NaN


## Everything You Must Know Cold

**.str accessor** — the bridge between Pandas and Python string methods. Without it, string methods don't work on a Series. With it, they apply to every element vectorized.

**str.strip()** — always do this first before any string comparison or groupby. Invisible whitespace is a silent killer.

**str.lower() before groupby** — *'Mumbai'* and *'mumbai'* create two separate groups. Standardize case first.

**str.contains(na=False)** — always pass *na=False* on columns that might have NaN. Without it you get a ValueError.

**str.split(expand=True)** — turns one structured column into multiple clean columns. One of the most powerful and commonly used string patterns.

**str.replace(regex=False)** — always set *regex=False* for simple text replacements. Avoids accidental regex interpretation of special characters like . or -.

**str.replace vs df.replace** — *str.replace* works inside the string (substring). *df.replace* replaces the whole cell value. Different tools for different jobs.